# Full simulation report (SYNTHETIC)

Component 1 · Anchor AI (J26-IT-366). All results here are **SYNTHETIC**. This is the whole component at work — the
MVD engine, Model 1 and Model 2 together — on the SYNTHETIC personas, for every seed and every variant, ending in the
same markdown report as `components/c1_disruption/sim/results/c1_simulation_SYNTHETIC.md`.

**Cell 1 — setup (run first).** In Google Colab this clones the C1 branch `dev/ravindu` of the team repo into
`/content/AnchorAI` **only if it is not there yet** (running it again never makes a nested copy), goes to that folder,
and installs only what these notebooks need (`components/c1_disruption/notebooks/requirements.txt`: numpy, scipy,
scikit-learn, quantile-forest, joblib, pandas, matplotlib — not the web server). It prints the versions. If a package
that was already loaded had to be replaced (usually numpy), it stops with a clear message: then use
**Runtime → Restart session** and run again from **cell 2** (the next code cell). Outside Colab it uses the repository
the notebook is in and installs nothing. C1 supports Python 3.11–3.13.

In [ ]:
import importlib.metadata as md, os, subprocess, sys

REPO = "https://github.com/DulanjaR/AnchorAI.git"
IN_COLAB = "google.colab" in sys.modules
PACKAGES = {"numpy": "numpy", "scipy": "scipy", "scikit-learn": "sklearn", "quantile-forest": "quantile_forest",
            "joblib": "joblib", "pandas": "pandas", "matplotlib": "matplotlib"}


def repo_root():
    if IN_COLAB:
        return "/content/AnchorAI"
    here = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(here, "components", "c1_disruption")):
        if os.path.dirname(here) == here:
            raise FileNotFoundError("Open this notebook inside the AnchorAI repository (or in Colab).")
        here = os.path.dirname(here)
    return here


def installed():
    versions = {}
    for package in PACKAGES:
        try:
            versions[package] = md.version(package)
        except md.PackageNotFoundError:
            versions[package] = None
    return versions


ROOT = repo_root()
if IN_COLAB and not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--depth", "1", "-b", "dev/ravindu", REPO, ROOT], check=True)
elif IN_COLAB:
    print("Already cloned at", ROOT, "- not cloning again.")
os.chdir(ROOT)
before = installed()
loaded = {p: getattr(sys.modules[m], "__version__", None) for p, m in PACKAGES.items() if m in sys.modules}
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    "components/c1_disruption/notebooks/requirements.txt"], check=True)
after = installed()
print("Python", sys.version.split()[0], "| folder", ROOT)
for package, version in after.items():
    print(f"  {package:16} {version}" + ("" if before[package] == version else f"   (was {before[package]})"))
replaced = [p for p, v in loaded.items() if v is not None and v != after[p]]
if replaced:
    print("\n" + "!" * 72 + f"\n{', '.join(replaced)} changed while already loaded.\n"
          "Runtime → Restart session, then run from cell 2 (the next code cell).\n" + "!" * 72)
    raise SystemExit("Restart needed: Runtime → Restart session, then run from cell 2.")

**Cell 2 — start here after a restart.** Goes to the repository folder, makes C1 importable, prints the versions in
use and starts the progress clock (every long step prints a line like `[  12.4s] training fold 3/5…`).

In [ ]:
import os, sys, tempfile

IN_COLAB = "google.colab" in sys.modules
ROOT = "/content/AnchorAI"
if not IN_COLAB:
    ROOT = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(ROOT, "components", "c1_disruption")):
        ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
# Files this notebook writes go here, never into the repository.
OUT = "/content/c1_outputs" if IN_COLAB else os.path.join(tempfile.gettempdir(), "c1_outputs")
os.makedirs(OUT, exist_ok=True)

import joblib, matplotlib, numpy, pandas, quantile_forest, scipy, sklearn
print("Python", sys.version.split()[0], "| numpy", numpy.__version__, "| scipy", scipy.__version__,
      "| scikit-learn", sklearn.__version__, "| quantile-forest", quantile_forest.__version__,
      "| joblib", joblib.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Outputs go to", OUT)
from components.c1_disruption.ml.progress import Progress
progress = Progress()

**Which data.** The SYNTHETIC persona files in `components/c1_disruption/sim/data/`: personas, their special habits,
30 days of planned tasks, each day's schedule and each persona's disruption profile (format:
`components/c1_disruption/sim/README.md`). Everything produced from them is **SYNTHETIC**: it checks that the pipeline
works; it is not a research finding.

In [ ]:
DATA = "components/c1_disruption/sim/data/"
PERSONAS, HABITS = DATA + "SYNTHETIC_personas.csv", DATA + "SYNTHETIC_habits.csv"
TASKS, SCHEDULE = DATA + "SYNTHETIC_tasks_planned.csv", DATA + "SYNTHETIC_schedule.csv"
PROFILE = DATA + "SYNTHETIC_disruption_profile.csv"
from components.c1_disruption.sim.personas import load_personas, load_schedule, load_tasks
from components.c1_disruption.sim.profile import load_profile
progress("loading the SYNTHETIC persona files…")
personas = load_personas(PERSONAS, HABITS)
tasks = load_tasks(TASKS, personas)
schedule = load_schedule(SCHEDULE, personas)
profile = load_profile(PROFILE, personas)
print(len(personas), "personas,", len(tasks), "planned tasks,", len(schedule), "scheduled days")

**All variants × seeds.** *planted* = persona drift as written; *null* = no drift (the model should not invent any);
*misspecified* = drift also depends on a hidden daily factor the model cannot see. Five seeds; every step prints a
progress line. The full run takes a while (about 20–40 minutes); set `QUICK = True` for one seed per variant.

In [ ]:
QUICK = False
from components.c1_disruption.sim import settings as sim_settings
from components.c1_disruption.sim.experiment import run, to_markdown
seeds = sim_settings.SEEDS[:1] if QUICK else sim_settings.SEEDS
report = run(PERSONAS, TASKS, seeds=seeds, schedule_csv=SCHEDULE, habits_csv=HABITS, profile_csv=PROFILE,
             progress=progress)
text = to_markdown(report)
report_file = os.path.join(OUT, "c1_simulation_SYNTHETIC.md")
open(report_file, "w", encoding="utf-8").write(text)
progress(f"report written to {report_file}")
print(text)

**Model 1 vs baselines, per variant** (mean over seeds).

In [ ]:
from statistics import mean
from components.c1_disruption.ml import charts
variants = list(dict.fromkeys(r["variant"] for r in report["runs"]))
for variant in variants:
    runs = [r for r in report["runs"] if r["variant"] == variant]
    methods = list(runs[0]["model1"]["warm"])
    charts.mae_by_method({m: {"cold start": mean(r["model1"]["cold"][m]["maeMinutes"] for r in runs),
                              "warm": mean(r["model1"]["warm"][m]["maeMinutes"] for r in runs)} for m in methods},
                         f"Model 1 vs baselines — {variant} (SYNTHETIC)");

**Learning curves** (first seed of each variant).

In [ ]:
first = {v: next(r for r in report["runs"] if r["variant"] == v) for v in variants}
charts.learning_curve({v: first[v]["model1"]["learningCurve"] for v in variants}, "Model 1 learning curves by variant (SYNTHETIC)");

**Predicted vs actual minutes** (planted variant, first seed, warm predictions).

In [ ]:
charts.predicted_vs_actual(first["planted"]["model1"]["scatter"], "Model 1: predicted vs actual minutes (SYNTHETIC, planted)");

**Earned-trust gate per persona**: share of seeds in which the gate was open at the persona's last warm point.

In [ ]:
for variant in variants:
    runs = [r for r in report["runs"] if r["variant"] == variant]
    share = {s: mean(float(r["model1"]["gateByStudent"][s]) for r in runs) for s in runs[0]["model1"]["gateByStudent"]}
    charts.gate_per_student(share, f"Earned-trust gate open per persona — {variant} (SYNTHETIC)");